# Quantization-aware training

Post-training quantization provide a reduction in size (4x) of models and an decrease in latency (2/3x). However, there is a trade-off, the accuracy is going to decrease: 

<img src="./images/01_a.png" width="500">

Quantization introduces information loss and therefore the inference accuracy from the quantized integer models are inevitably lower than that from the floating point models. Such information loss is due to that the floating points after quantization and de-quantization is not exactly recoverable. Mathematically, it means:

$\begin{align}
x = f_d(f_q(x,s_x,z_x),s_x,z_x) + \Delta_x
\end{align}$

If $\Delta_x=0$, the inference accuracy of the quantized model would be exactly the same as the  accuracy from the floating point model. Unfortunately, it is not. 

Is there anything we can do to improve the accuracy while keeping the benefits that we get from post-training quantization? Notice that conversion mechanism is chopping off some bit of information that the floating-point values have (thanks to a much wider range). The idea is to inform the network about this source error in order to allow the network to learn how to become resilient against that error.

The idea of **quantization aware training** is to ask the neural network to take the effect of such information loss into account during training. Therefore, we add a quantization and a de-quantization layer for each tensor. Mathematically, it means:

$\begin{align}
\hat{x} = D(Q(x,s_x,z_x),s_x,z_x) = s_x(\text{clip}(\text{round}(\frac{x}{s_x}+z_x),max_q,min_q)-z_x)
\end{align}$

the scale and zero point could be estimated using the same method of static quantization. In that way, all the data types for the quantized tensors are still floating point tensors and the training is supposed to be done normally as if the quantization and de-quantization layers were not existed. In this way we inject that error into the training pipeline, so that the network naturally learns to become resilient against it.

Let's take a look at what happens when we are comparing post-training quantization versus quantization-aware training:

<img src="./images/03_a.png" width="500">

Quantization-aware training is not guaranteed to always be able to do better (it mostly do better).This is part of the black box magic with machine learning. This is another hyperparameter that we have to think about. Another interesting thing is that sometimes it does even better that the original floating-point network. We cannot control exactly what's happening in quantization because when we are doing quantization-aware training, the network is **learning differently** from the way it originally learned.

Let's start with a convolutional neural network that classifies the MNIST handwriting digits and as an experiment, train it for some epochs.

In [1]:
from tensorflow import keras

mnist = keras.datasets.mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

train_images = train_images / 255.0
test_images = test_images / 255.0

2023-05-25 08:49:18.394029: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
model = keras.Sequential([
  keras.layers.InputLayer(input_shape=(28, 28)),
  keras.layers.Reshape(target_shape=(28, 28, 1)),
  keras.layers.Conv2D(filters=12, kernel_size=(3, 3), activation='relu'),
  keras.layers.MaxPooling2D(pool_size=(2, 2)),
  keras.layers.Flatten(),
  keras.layers.Dense(10)
])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 12)        120       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 13, 13, 12)       0         
 )                                                               
                                                                 
 flatten (Flatten)           (None, 2028)              0         
                                                                 
 dense (Dense)               (None, 10)                20290     
                                                                 
Total params: 20,410
Trainable params: 20,410
Non-trainable params: 0
____________________________________________________

In [3]:
import tensorflow as tf

model.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

results = model.fit(train_images, train_labels, epochs=5, validation_split=0.1, verbose=1)

print('Training Accuracy: ', results.history['accuracy'][4])
print('Validation Accuracy: ', results.history['val_accuracy'][4])

Epoch 1/5
1688/1688 [==============================] - 6s 3ms/step - loss: 0.2880 - accuracy: 0.9192 - val_loss: 0.1206 - val_accuracy: 0.9670
Epoch 2/5
1688/1688 [==============================] - 6s 3ms/step - loss: 0.1206 - accuracy: 0.9651 - val_loss: 0.0873 - val_accuracy: 0.9757
Epoch 3/5
1688/1688 [==============================] - 6s 3ms/step - loss: 0.0924 - accuracy: 0.9728 - val_loss: 0.0802 - val_accuracy: 0.9785
Epoch 4/5
1688/1688 [==============================] - 6s 3ms/step - loss: 0.0765 - accuracy: 0.9776 - val_loss: 0.0728 - val_accuracy: 0.9807
Epoch 5/5
1688/1688 [==============================] - 7s 4ms/step - loss: 0.0657 - accuracy: 0.9801 - val_loss: 0.0665 - val_accuracy: 0.9823
Training Accuracy:  0.9800740480422974
Validation Accuracy:  0.9823333621025085


We got accuracy of 99% on the training set and 98% on the validation set. We use it as our baseline. We can convert the model in TFLite without quantization and save it in order to make some comparisons.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
float_tflite_model = converter.convert()

tf.saved_model.save(float_tflite_model, "./models/float_model")

In order to quantize the model, we can use the [TensorFlow Model Optimization toolkit](https://www.tensorflow.org/model_optimization) and passing in input the Keras model. What that API is doing is extending that network with the ability to mimic the quantized behavior that would be happening during the inference time, during the training time:

In [ ]:
import tensorflow_model_optimization as tfmot

quantize_model = tfmot.quantization.keras.quantize_model

q_aware_model = quantize_model(model)

q_aware_model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

q_aware_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 quantize_layer_2 (QuantizeL  (None, 28, 28)           3         
 ayer)                                                           
                                                                 
 quant_reshape_1 (QuantizeWr  (None, 28, 28, 1)        1         
 apperV2)                                                        
                                                                 
 quant_conv2d_1 (QuantizeWra  (None, 26, 26, 12)       147       
 pperV2)                                                         
                                                                 
 quant_max_pooling2d_1 (Quan  (None, 13, 13, 12)       1         
 tizeWrapperV2)                                                  
                                                                 
 quant_flatten_1 (QuantizeWr  (None, 2028)            

When we train the network, it is implicitly learning to be resilient to the quantization error.

In [ ]:
results = q_aware_model.fit(train_images, train_labels, epochs=5, validation_split=0.1, verbose=1)

print('Training Accuracy: ', results.history['accuracy'][4])
print('Validation Accuracy: ', results.history['val_accuracy'][4])

Epoch 1/5
1688/1688 [==============================] - 9s 5ms/step - loss: 0.0530 - accuracy: 0.9839 - val_loss: 0.0631 - val_accuracy: 0.9840
Epoch 2/5
1688/1688 [==============================] - 9s 5ms/step - loss: 0.0467 - accuracy: 0.9861 - val_loss: 0.0604 - val_accuracy: 0.9843
Epoch 3/5
1688/1688 [==============================] - 9s 6ms/step - loss: 0.0421 - accuracy: 0.9873 - val_loss: 0.0582 - val_accuracy: 0.9845
Epoch 4/5
1688/1688 [==============================] - 10s 6ms/step - loss: 0.0379 - accuracy: 0.9881 - val_loss: 0.0567 - val_accuracy: 0.9855
Epoch 5/5
1688/1688 [==============================] - 11s 7ms/step - loss: 0.0345 - accuracy: 0.9893 - val_loss: 0.0571 - val_accuracy: 0.9850
Training Accuracy:  0.9892963171005249
Validation Accuracy:  0.9850000143051147


In the example, there is no loss in test accuracy after quantization aware training, compared to the baseline. You can try this for yourself, training for more epochs to see the impact.

In [ ]:
_, baseline_model_accuracy = model.evaluate(test_images, test_labels, verbose=0)
_, q_aware_model_accuracy = q_aware_model.evaluate(test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)
print('Quantized test accuracy:', q_aware_model_accuracy)

Baseline test accuracy: 0.9805999994277954
Quantized test accuracy: 0.9815000295639038


After this, you have a quantized model that we can convert in TFLite format:

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

quantized_tflite_model = converter.convert()

tf.saved_model.save(quantized_tflite_model, "./models/quantized_model")

We can consider the persistence of accuracy from TF to TFLite. To this end we define a helper function to evaluate the TF Lite model on the test dataset:

In [ ]:
import numpy as np

def evaluate_model(interpreter, test_images):
    interpreter.allocate_tensors()
    input_index = interpreter.get_input_details()[0]["index"]
    output_index = interpreter.get_output_details()[0]["index"]

    prediction_digits = []
    for i, test_image in enumerate(test_images):
        
        # add batch dimension and convert to float32 to match with the model's input data format.
        test_image = np.expand_dims(test_image, axis=0).astype(np.float32)
        interpreter.set_tensor(input_index, test_image)
        
        interpreter.invoke()

        # remove batch dimension and find the digit with highest probability.
        output = interpreter.tensor(output_index)
        digit = np.argmax(output()[0])
        prediction_digits.append(digit)

    prediction_digits = np.array(prediction_digits)
    accuracy = (prediction_digits == test_labels).mean()
    return accuracy

We evaluate the quantized model:

In [ ]:
interpreter = tf.lite.Interpreter(model_content=quantized_tflite_model)

quantized_model_accuracy = evaluate_model(interpreter, test_images)

print('Quantized TFLite model test accuracy:', quantized_model_accuracy)
print('Quantization-aware TF model test accuracy:', q_aware_model_accuracy)

Quantized TFLite model test accuracy: 0.9814
Quantization-aware TF model test accuracy: 0.9815000295639038


We can consider also the reduction of model size after quantization:

In [ ]:
import os

print("Float model in MB:", os.path.getsize('./models/float_model.tflite') / float(2**20))
print("Quantized model in MB:", os.path.getsize('./models/quantized_model.tflite') / float(2**20))

Float model in Mb: 0.08089065551757812
Quantized model in Mb: 0.0238037109375
